In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
!pip -q install pandas pyarrow fastparquet gcsfs google-cloud-storage requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 17.5 MB/s eta 0:00:00


In [ ]:
from google.cloud import storage

In [ ]:
PROJECT_ID = "project-8e2366a6-d3cc-40ee-9de"

BUCKET_NAME = "project-8e2366a6-d3cc-40ee-9de-bronze-raw-dev"

### **Ingestion POS: danh sách cơ sở y tế được chứng nhận Medicare**

In [ ]:
import requests
import pandas as pd

CMS_POS_URL = (
    "https://data.cms.gov/data-api/v1/dataset/"
    "8f6da2b1-f719-40c2-8f73-2b7ecb7fb42a/data"  ## mã id của từng quý, cập nhật để lấy hết bộ dữ liệu
)

response = requests.get(
    CMS_POS_URL,
    params={
        "limit": 500000
    }
)

data = response.json()

df_pos = pd.DataFrame(data)

print(df_pos.shape)
df_pos.head()

(1000, 473)


,PRVDR_CTGRY_SBTYP_CD,PRVDR_CTGRY_CD,CHOW_CNT,CHOW_DT,CITY_NAME,ACPTBL_POC_SW,CMPLNC_STUS_CD,SSA_CNTY_CD,CROSS_REF_PROVIDER_NUMBER,CRTFCTN_DT,...,OTHR_SRGRY_SW,PAIN_SRGRY_SW,PLSTC_SRGRY_SW,FT_SRGRY_SW,SB_SW,SB_SIZE_CD,TCHNLGST_2_YR_RDLGC_CNT,TCHNLGST_ASCT_DGR_CNT,TCHNLGST_BS_BA_DGR_CNT,DLYS_STN_CNT
0,01,01,1,,DOTHAN,N,B,340,010163,20171005,...,,,,,N,,,,,
1,01,01,0,,BRIDGEPORT,N,A,350,,20010316,...,,,,,N,,,,,
2,01,01,0,,BOAZ,Y,A,470,,20021003,...,,,,,N,,,,,
3,01,01,1,20100701,FLORENCE,N,B,380,,20190411,...,,,,,N,,,,,
4,01,01,0,,OPP,N,B,190,,20190523,...,,,,,Y,2,,,,


In [ ]:
## chuyển đổi csv sang parquet để đẩy lên gcs
import os

os.makedirs("data", exist_ok=True)

pos_parquet_path = "data/cms_pos.parquet"

df_pos.to_parquet(
    pos_parquet_path,
    engine="pyarrow",
    index=False
)

print("Saved:", pos_parquet_path)

Saved: data/cms_pos.parquet


In [ ]:
## đẩy lên gcs

client = storage.Client(project=PROJECT_ID)

bucket = client.bucket(BUCKET_NAME)

blob = bucket.blob(
    "cms/pos/cms_pos_q1_2020.parquet"
)
blob.upload_from_filename(pos_parquet_path)

print("Uploaded POS parquet to GCS")

Uploaded POS parquet to GCS


In [ ]:
import pandas as pd

# 1. Xem thử 5 dòng đầu
print(df_pos.head())

# 2. Xem tổng quan về các cột và kiểu dữ liệu
print(df_pos.info())

# 3. Thống kê nhanh cho các cột số
print(df_pos.describe())

# 4. Đếm số lượng nhà cung cấp theo từng bang
print(df_pos['STATE_CD'].value_counts())

# 5. Lọc ra các cơ sở đã chấm dứt hoạt động
df_terminated = df_pos[df_pos['TRMNTN_EXPRTN_DT'].notna()]
print(f"Số cơ sở đã chấm dứt: {len(df_terminated)}")

  PRVDR_CTGRY_SBTYP_CD PRVDR_CTGRY_CD CHOW_CNT   CHOW_DT   CITY_NAME  \
0                   01             01        1                DOTHAN   
1                   01             01        0            BRIDGEPORT   
2                   01             01        0                  BOAZ   
3                   01             01        1  20100701    FLORENCE   
4                   01             01        0                   OPP   

  ACPTBL_POC_SW CMPLNC_STUS_CD SSA_CNTY_CD CROSS_REF_PROVIDER_NUMBER  \
0             N              B         340                    010163   
1             N              A         350                             
2             Y              A         470                             
3             N              B         380                             
4             N              B         190                             

  CRTFCTN_DT  ... OTHR_SRGRY_SW PAIN_SRGRY_SW PLSTC_SRGRY_SW FT_SRGRY_SW  \
0   20171005  ...                                         

### **Ingestion data từ HHS: COVID-19 Reported Patient Impact and Hospital Capacity by State (RAW)**
Thời gian: tháng 01/2020 đến tháng 05/2024.


In [ ]:
!pip install sodapy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sodapy import Socrata

client = Socrata(
    "healthdata.gov",
    None
)
DATASET_ID = "anag-cw7u"
results = client.get(
    DATASET_ID,
    limit=50000
)

df_hhs = pd.DataFrame.from_records(results)
# HHS_URL = "https://healthdata.gov/resource/6xf2-c3ie.csv"

# df_hhs = pd.read_csv(
#     HHS_URL
# )

print(df_hhs.shape)
print("Ngày bắt đầu:", df_hhs['collection_week'].min())
print("Ngày kết thúc:", df_hhs['collection_week'].max())
df_hhs.head()

(50000, 129)
Ngày bắt đầu: 2019-12-29T00:00:00.000
Ngày kết thúc: 2024-04-21T00:00:00.000


,hospital_pk,collection_week,state,ccn,hospital_name,address,city,zip,hospital_subtype,fips_code,...,previous_day_admission_pediatric_covid_confirmed_5_11_7_day_sum,previous_day_admission_pediatric_covid_confirmed_unknown_7_day_sum,staffed_icu_pediatric_patients_confirmed_covid_7_day_avg,staffed_icu_pediatric_patients_confirmed_covid_7_day_sum,previous_week_personnel_covid_vaccinated_doses_administered_7_day,total_personnel_covid_vaccinated_doses_none_7_day,total_personnel_covid_vaccinated_doses_one_7_day,total_personnel_covid_vaccinated_doses_all_7_day,previous_week_patients_covid_vaccinated_doses_one_7_day,previous_week_patients_covid_vaccinated_doses_all_7_day
0,370094,2021-01-10T00:00:00.000,OK,370094,SSM HEALTH ST ANTHONY HOSPITAL - MIDWEST,2825 PARKLAWN DRIVE,MIDWEST CITY,73110,Short Term,40109,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,370220,2020-06-14T00:00:00.000,OK,370220,ONECORE HEALTH,100 NE 85TH STREET,OKLAHOMA CITY,73114,Short Term,40109,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,431338,2022-12-25T00:00:00.000,SD,431338,AVERA GREGORY HOSPITAL,110 S LOGAN AVE,GREGORY,57533,Critical Access Hospitals,46053,...,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,450424,2020-10-18T00:00:00.000,TX,450424,HOUSTON METHODIST BAYTOWN HOSPITAL,4401 GARTH ROAD,BAYTOWN,77521,Short Term,48201,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,453316,2020-08-02T00:00:00.000,TX,453316,CHILDRENS MEDICAL CENTER PLANO,7601 PRESTON ROAD,PLANO,75024,Childrens Hospitals,48085,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(list(df_hhs.columns))

['hospital_pk', 'collection_week', 'state', 'ccn', 'hospital_name', 'address', 'city', 'zip', 'hospital_subtype', 'fips_code', 'is_metro_micro', 'total_beds_7_day_avg', 'all_adult_hospital_beds_7_day_avg', 'all_adult_hospital_inpatient_beds_7_day_avg', 'inpatient_beds_used_7_day_avg', 'all_adult_hospital_inpatient_bed_occupied_7_day_avg', 'inpatient_beds_used_covid_7_day_avg', 'total_adult_patients_hospitalized_confirmed_and_suspected_covid_7_day_avg', 'total_adult_patients_hospitalized_confirmed_covid_7_day_avg', 'total_pediatric_patients_hospitalized_confirmed_and_suspected_covid_7_day_avg', 'total_pediatric_patients_hospitalized_confirmed_covid_7_day_avg', 'inpatient_beds_7_day_avg', 'total_icu_beds_7_day_avg', 'total_staffed_adult_icu_beds_7_day_avg', 'icu_beds_used_7_day_avg', 'staffed_adult_icu_bed_occupancy_7_day_avg', 'staffed_icu_adult_patients_confirmed_and_suspected_covid_7_day_avg', 'staffed_icu_adult_patients_confirmed_covid_7_day_avg', 'total_patients_hospitalized_confirm

In [ ]:
KEEP_COLS = [
    "hospital_pk",
    "collection_week",
    "state",
    "ccn",
    "hospital_name",
    "address",
    "city",
    "zip",
    "hospital_subtype",
    "fips_code",
    "is_metro_micro",

    "total_beds_7_day_avg",
    "all_adult_hospital_beds_7_day_avg",
    "all_adult_hospital_inpatient_beds_7_day_avg",
    "inpatient_beds_used_7_day_avg",
    "all_adult_hospital_inpatient_bed_occupied_7_day_avg",
    "inpatient_beds_used_covid_7_day_avg",

    "total_adult_patients_hospitalized_confirmed_and_suspected_covid_7_day_avg",
    "total_adult_patients_hospitalized_confirmed_covid_7_day_avg",

    "total_pediatric_patients_hospitalized_confirmed_and_suspected_covid_7_day_avg",
    "total_pediatric_patients_hospitalized_confirmed_covid_7_day_avg",

    "inpatient_beds_7_day_avg",
    "total_icu_beds_7_day_avg",
    "total_staffed_adult_icu_beds_7_day_avg",
    "icu_beds_used_7_day_avg",
    "staffed_adult_icu_bed_occupancy_7_day_avg",

    "staffed_icu_adult_patients_confirmed_and_suspected_covid_7_day_avg",
    "staffed_icu_adult_patients_confirmed_covid_7_day_avg",

    "total_patients_hospitalized_confirmed_influenza_7_day_avg",
    "icu_patients_confirmed_influenza_7_day_avg",

    "previous_day_admission_adult_covid_confirmed_7_day_sum",
    "previous_day_admission_pediatric_covid_confirmed_7_day_sum",
    "previous_day_total_ed_visits_7_day_sum",

    "geocoded_hospital_address",
    "hhs_ids",

    "is_corrected",

    "all_pediatric_inpatient_bed_occupied_7_day_avg",
    "all_pediatric_inpatient_beds_7_day_avg",

    "staffed_pediatric_icu_bed_occupancy_7_day_avg",

    "total_staffed_pediatric_icu_beds_7_day_avg",

    "staffed_icu_pediatric_patients_confirmed_covid_7_day_avg",

    "previous_week_personnel_covid_vaccinated_doses_administered_7_day",

    "total_personnel_covid_vaccinated_doses_none_7_day",
    "total_personnel_covid_vaccinated_doses_one_7_day",
    "total_personnel_covid_vaccinated_doses_all_7_day",

    "previous_week_patients_covid_vaccinated_doses_one_7_day",
    "previous_week_patients_covid_vaccinated_doses_all_7_day"

]
df_hhs = df_hhs[KEEP_COLS]

In [ ]:
from dataclasses import dataclass
from typing import List
from datetime import datetime
from dataclasses import dataclass
from typing import List

@dataclass
class ValidationResult:
    passed: bool
    errors: List[str]
    warnings: List[str]

In [ ]:
def validate_hhs_rows(df: pd.DataFrame):

    df = df.copy()

    errors_col = []

    required_cols = [
        "hospital_pk",
        "state",
        "collection_week"
    ]

    numeric_cols = [
        "inpatient_beds_7_day_avg",
        "total_icu_beds_7_day_avg",
        "inpatient_beds_used_7_day_avg"
    ]

    for idx, row in df.iterrows():

        row_errors = []

        # ====================================
        # Rule 1: Required columns not null
        # ====================================

        for col in required_cols:

            if col not in df.columns:
                row_errors.append(f"missing_column:{col}")

            elif pd.isna(row[col]):

                row_errors.append(f"null:{col}")

        # ====================================
        # Rule 2: Numeric columns >= 0
        # ====================================

        for col in numeric_cols:

            if col in df.columns:

                val = pd.to_numeric(
                    row[col],
                    errors="coerce"
                )

                if pd.notna(val) and val < 0:

                    row_errors.append(f"negative:{col}")

        errors_col.append(row_errors)

    df["_validation_errors"] = errors_col

    # valid rows
    valid_df = df[
        df["_validation_errors"].str.len() == 0
    ].copy()

    # invalid rows
    invalid_df = df[
        df["_validation_errors"].str.len() > 0
    ].copy()

    return valid_df, invalid_df

In [ ]:
valid_df, invalid_df = validate_hhs_rows(df_hhs)
print("VALID:", len(valid_df))
print("INVALID:", len(invalid_df))
invalid_df[
    [
        "hospital_pk",
        "_validation_errors"
    ]
].head()

VALID: 43811
INVALID: 6189


,hospital_pk,_validation_errors
2,431338,"[negative:total_icu_beds_7_day_avg, negative:i..."
12,451379,[negative:total_icu_beds_7_day_avg]
16,451320,[negative:inpatient_beds_used_7_day_avg]
42,380001,[negative:total_icu_beds_7_day_avg]
47,371322,[negative:inpatient_beds_used_7_day_avg]


In [ ]:
valid_parquet_path = "data/hhs_capacity_valid.parquet"

invalid_parquet_path = "data/hhs_capacity_invalid.parquet"

In [ ]:

valid_df.to_parquet(
    valid_parquet_path,
    engine="pyarrow",
    index=False
)

In [ ]:
valid_blob = bucket.blob(
    f"hhs/hospital_capacity/hhs_capacity.parquet"
)

valid_blob.upload_from_filename(
    valid_parquet_path
)

print("Uploaded VALID parquet")

Uploaded VALID parquet


In [ ]:
quarantine_bucket = client.bucket(
    "project-8e2366a6-d3cc-40ee-9de-quarantine-dev"
)
invalid_blob = quarantine_bucket.blob(
    f"hhs/hospital_capacity/hhs_capacity_invalid.parquet"
)
invalid_df.to_parquet(
    invalid_parquet_path,
    engine="pyarrow",
    index=False
)
invalid_blob.upload_from_filename(
    invalid_parquet_path
)

print("Uploaded INVALID parquet")

Uploaded INVALID parquet


### Step: ETL ###
**Chuyển sang notebook mới**
